In [1]:
import os
import zipfile
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

RANDOM_STATE = 42
THRESHOLD = 0.9

print("라이브러리 로드 완료")

라이브러리 로드 완료


In [2]:
import os
import pandas as pd
import numpy as np

DATA_PATH = "data/fraud_full_features.csv"  # BDAI/ 루트에서 실행 기준(팀 공통 경로)
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join("..", DATA_PATH)  # Modeling/ 등 하위 폴더에서 실행한 경우 보정

# CSV 불러오기
df = pd.read_csv(DATA_PATH)

print("데이터 크기:", df.shape)
print("컬럼 수:", len(df.columns))

데이터 크기: (1296675, 31)
컬럼 수: 31


In [3]:
# ==========================================
# STEP 5. 데이터 기본 확인
# ==========================================

print("데이터 크기:", df.shape)

print("\n상위 5개 행:")
display(df.head())

print("\n타깃 분포:")
print(df["is_fraud"].value_counts())

print(
    "\nFraud Rate (%):",
    df["is_fraud"].mean() * 100
)

데이터 크기: (1296675, 31)

상위 5개 행:


,trans_date_trans_time,cc_num,merchant,category,amt,is_fraud,recent_24h_high_amt_count,category_recent_fraud_rate,category_recent_fraud_rate_missing,count_30min,Repeat3,high_speed,speed_2,customer_mean_amt,customer_std_amt,amt_ratio_to_mean,amt_zscore_card,customer_transaction_count,trans_hour,age,prior_normal_median_amt,amt_to_prior_median_ratio,is_10x_prior_median,has_prior_normal_transaction,outside_trans_hours_80,is_online,risk_time_22_04,interact_repeat_category,merchant_change_count,rolling_sum_amt_1h,is_high_amt
0,2019-01-01 12:47:15,60416207185,"fraud_Jones, Sawayn and Romaguera",misc_net,7.270000,0,0,0.000000,0,0,0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0,12,32,NaN,NaN,0,0,0,1,0,0.000000,0,7.270000,0
1,2019-01-02 08:44:57,60416207185,fraud_Berge LLC,gas_transport,52.940000,0,0,0.006154,0,1,0,0,0.000000,7.270000,0.000000,7.281981,0.000000,1,8,32,7.270000,7.281981,0,1,0,0,0,0.000000,1,52.940000,0
2,2019-01-02 08:47:36,60416207185,fraud_Luettgen PLC,gas_transport,82.080000,0,0,0.006135,0,2,0,1,2382.349553,30.105000,32.293567,2.726457,1.609454,2,8,32,30.105000,2.726457,0,1,0,0,0,0.000000,1,135.020000,0
3,2019-01-02 12:38:14,60416207185,fraud_Daugherty LLC,kids_pets,34.790000,0,0,0.000000,0,1,0,0,22.933099,47.430000,37.708144,0.733502,-0.335206,3,12,32,52.940000,0.657159,0,1,0,0,0,0.000000,1,34.790000,0
4,2019-01-02 13:10:46,60416207185,fraud_Beier and Sons,home,27.180000,0,0,0.000000,0,1,0,1,245.059622,44.270000,31.430534,0.613960,-0.543739,4,13,32,43.865000,0.619628,0,1,0,0,0,0.000000,1,61.970000,0



타깃 분포:
is_fraud
0    1289169
1       7506
Name: count, dtype: int64

Fraud Rate (%): 0.5788651743883394


In [4]:
# ==========================================
# STEP 6. Feature Set 1~5 정의
# ==========================================

set1 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min"
]


set2 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",
    "merchant_change_count"
]


set3 = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",
    "high_speed"
]


set4 = [
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "category",
    "amt",
    "trans_hour",
    "age"
]


set5 = [
    "category",
    "amt",
    "is_online",
    "recent_24h_high_amt_count",
    "category_recent_fraud_rate",
    "speed_2",
    "customer_mean_amt",
    "customer_std_amt",
    "amt_ratio_to_mean",
    "amt_zscore_card",
    "customer_transaction_count",
    "trans_hour",
    "age",
    "rolling_sum_amt_1h",
    "prior_normal_median_amt",
    "amt_to_prior_median_ratio",
    "risk_time_22_04",
    "interact_repeat_category",
    "has_prior_normal_transaction"
]


feature_sets = {
    "Set 1": set1,
    "Set 2": set2,
    "Set 3": set3,
    "Set 4": set4,
    "Set 5": set5
}


for name, features in feature_sets.items():
    print(
        f"{name}: {len(features)}개 변수"
    )

Set 1: 10개 변수
Set 2: 11개 변수
Set 3: 11개 변수
Set 4: 6개 변수
Set 5: 19개 변수


In [5]:
# ==========================================
# STEP 7. 변수 존재 여부 확인
# ==========================================

print("===== 변수 존재 여부 확인 =====")

for set_name, features in feature_sets.items():

    missing = [
        feature
        for feature in features
        if feature not in df.columns
    ]

    if len(missing) == 0:

        print(
            f"✅ {set_name}: "
            f"모든 변수 존재 ({len(features)}개)"
        )

    else:

        print(
            f"❌ {set_name}: "
            f"누락 변수 = {missing}"
        )

===== 변수 존재 여부 확인 =====
✅ Set 1: 모든 변수 존재 (10개)
✅ Set 2: 모든 변수 존재 (11개)
✅ Set 3: 모든 변수 존재 (11개)
✅ Set 4: 모든 변수 존재 (6개)
✅ Set 5: 모든 변수 존재 (19개)


In [6]:
# ==========================================
# STEP 8. 결측치 확인
# ==========================================

all_model_features = sorted(
    set(
        feature
        for features in feature_sets.values()
        for feature in features
    )
)

missing_summary = (
    df[all_model_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("===== 결측치가 있는 변수 =====")

display(
    missing_summary[
        missing_summary > 0
    ]
)

===== 결측치가 있는 변수 =====


amt_to_prior_median_ratio    1649
prior_normal_median_amt      1649
dtype: int64

In [7]:
# ==========================================
# STEP 9. 전처리 + Logistic Regression
# ==========================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


CATEGORICAL_COLUMNS = [
    "category"
]


def make_logistic_pipeline(
    features,
    C=1.0,
    penalty="l2",
    solver="lbfgs"
):

    categorical_features = [
        col
        for col in features
        if col in CATEGORICAL_COLUMNS
    ]

    numeric_features = [
        col
        for col in features
        if col not in CATEGORICAL_COLUMNS
    ]


    numeric_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ])


    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            )
        )
    ])


    preprocessor = ColumnTransformer([
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ])


    model = LogisticRegression(
        C=C,
        penalty=penalty,
        solver=solver,
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )


    pipeline = Pipeline([
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            model
        )
    ])

    return pipeline

In [8]:
# ==========================================
# STEP 10. 평가 함수
# ==========================================

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

THRESHOLD = 0.9


def evaluate_binary_model(
    y_true,
    prob,
    threshold=0.9
):

    pred = (
        prob >= threshold
    ).astype(int)


    tn, fp, fn, tp = confusion_matrix(
        y_true,
        pred,
        labels=[0, 1]
    ).ravel()


    return {

        "PR-AUC":
            average_precision_score(
                y_true,
                prob
            ),

        "ROC-AUC":
            roc_auc_score(
                y_true,
                prob
            ),

        "Accuracy":
            accuracy_score(
                y_true,
                pred
            ),

        "Precision":
            precision_score(
                y_true,
                pred,
                zero_division=0
            ),

        "Recall":
            recall_score(
                y_true,
                pred,
                zero_division=0
            ),

        "F1":
            f1_score(
                y_true,
                pred,
                zero_division=0
            ),

        "FP": fp,
        "FN": fn,
        "TP": tp,
        "TN": tn
    }

In [9]:
# ==========================================
# STEP 11. 시간순 80:20 성능 비교
# ==========================================

results_80 = []

# 시간순 80:20 분할
split_idx_80 = int(len(df) * 0.8)

train_df_80 = df.iloc[:split_idx_80].copy()
val_df_20 = df.iloc[split_idx_80:].copy()

print("===== 시간순 80:20 분할 =====")
print("Train:", train_df_80.shape)
print("Validation:", val_df_20.shape)

print("\nTrain 기간:")
print(
    train_df_80["trans_date_trans_time"].min(),
    "~",
    train_df_80["trans_date_trans_time"].max()
)

print("\nValidation 기간:")
print(
    val_df_20["trans_date_trans_time"].min(),
    "~",
    val_df_20["trans_date_trans_time"].max()
)

print("\nTrain Fraud Rate:")
print(
    train_df_80["is_fraud"].mean() * 100,
    "%"
)

print("\nValidation Fraud Rate:")
print(
    val_df_20["is_fraud"].mean() * 100,
    "%"
)


# ==========================================
# Feature Set 1~5 학습
# ==========================================

for set_name, features in feature_sets.items():

    print("\n" + "=" * 70)
    print(f"80:20 | {set_name}")
    print("=" * 70)

    X_train = train_df_80[features]
    y_train = train_df_80["is_fraud"]

    X_val = val_df_20[features]
    y_val = val_df_20["is_fraud"]

    model = make_logistic_pipeline(
        features=features,
        C=1.0,
        penalty="l2",
        solver="lbfgs"
    )

    model.fit(
        X_train,
        y_train
    )

    val_prob = model.predict_proba(
        X_val
    )[:, 1]

    metrics = evaluate_binary_model(
        y_true=y_val,
        prob=val_prob,
        threshold=THRESHOLD
    )

    results_80.append({
        "Feature Set": set_name,
        "N Features": len(features),
        "Threshold": THRESHOLD,
        **metrics
    })

    print(f"PR-AUC    : {metrics['PR-AUC']:.6f}")
    print(f"ROC-AUC   : {metrics['ROC-AUC']:.6f}")
    print(f"Precision : {metrics['Precision']:.6f}")
    print(f"Recall    : {metrics['Recall']:.6f}")
    print(f"F1        : {metrics['F1']:.6f}")
    print(
        f"TP={metrics['TP']:,} | "
        f"FP={metrics['FP']:,} | "
        f"FN={metrics['FN']:,} | "
        f"TN={metrics['TN']:,}"
    )

===== 시간순 80:20 분할 =====
Train: (1037340, 31)
Validation: (259335, 31)

Train 기간:
2019-01-01 00:00:18 ~ 2020-06-21 12:13:36

Validation 기간:


2019-01-01 00:05:08 ~ 2020-06-21 12:13:37

Train Fraud Rate:
0.5739680336244626 %

Validation Fraud Rate:
0.5984537374438468 %

80:20 | Set 1


PR-AUC    : 0.448847
ROC-AUC   : 0.968048
Precision : 0.338626
Recall    : 0.755799
F1        : 0.467703
TP=1,173 | FP=2,291 | FN=379 | TN=255,492

80:20 | Set 2


PR-AUC    : 0.450859
ROC-AUC   : 0.968809
Precision : 0.342890
Recall    : 0.762887
F1        : 0.473127
TP=1,184 | FP=2,269 | FN=368 | TN=255,514

80:20 | Set 3


PR-AUC    : 0.446447
ROC-AUC   : 0.967986
Precision : 0.338435
Recall    : 0.755155
F1        : 0.467398
TP=1,172 | FP=2,291 | FN=380 | TN=255,492

80:20 | Set 4


PR-AUC    : 0.420030
ROC-AUC   : 0.966964
Precision : 0.300469
Recall    : 0.742268
F1        : 0.427776
TP=1,152 | FP=2,682 | FN=400 | TN=255,101

80:20 | Set 5


PR-AUC    : 0.525199
ROC-AUC   : 0.982593
Precision : 0.407821
Recall    : 0.799613
F1        : 0.540152
TP=1,241 | FP=1,802 | FN=311 | TN=255,981


In [10]:
# ==========================================
# STEP 12. 80:20 성능 순위
# ==========================================

results_80_df = pd.DataFrame(
    results_80
)

results_80_df = (
    results_80_df
    .sort_values(
        by=[
            "PR-AUC",
            "F1",
            "Recall"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

results_80_df.insert(
    0,
    "Rank",
    range(1, len(results_80_df) + 1)
)

display(
    results_80_df[
        [
            "Rank",
            "Feature Set",
            "N Features",
            "PR-AUC",
            "Precision",
            "Recall",
            "F1",
            "FP",
            "FN"
        ]
    ]
)

,Rank,Feature Set,N Features,PR-AUC,Precision,Recall,F1,FP,FN
0,1,Set 5,19,0.525199,0.407821,0.799613,0.540152,1802,311
1,2,Set 2,11,0.450859,0.342890,0.762887,0.473127,2269,368
2,3,Set 1,10,0.448847,0.338626,0.755799,0.467703,2291,379
3,4,Set 3,11,0.446447,0.338435,0.755155,0.467398,2291,380
4,5,Set 4,6,0.420030,0.300469,0.742268,0.427776,2682,400


In [11]:
# ==========================================
# STEP 13. 3-Fold 경계 (다영/민정과 동일한 행 개수 6등분 방식)
# ==========================================

boundaries = np.linspace(0, len(df), 7, dtype=int)

folds = {}
for fold_number in range(1, 4):
    train_end = boundaries[fold_number + 2]
    val_start = train_end
    val_end = boundaries[fold_number + 3]
    folds[f'Fold {fold_number}'] = {
        'train_end': train_end,
        'val_start': val_start,
        'val_end': val_end,
    }


def get_fold_data(data, period):
    fold_train = data.iloc[0:period['train_end']].copy()
    fold_val = data.iloc[period['val_start']:period['val_end']].copy()
    return fold_train, fold_val


In [12]:
fold_check = []

for fold_name, period in folds.items():

    fold_train, fold_val = get_fold_data(
        df,
        period
    )

    fold_check.append({

        "Fold": fold_name,

        "Train Transactions":
            len(fold_train),

        "Train Fraud":
            int(
                fold_train["is_fraud"].sum()
            ),

        "Validation Transactions":
            len(fold_val),

        "Validation Fraud":
            int(
                fold_val["is_fraud"].sum()
            ),

        "Validation Fraud Rate (%)":
            (
                fold_val["is_fraud"].mean()
                * 100
            )
    })


fold_check_df = pd.DataFrame(
    fold_check
)

display(
    fold_check_df
)

,Fold,Train Transactions,Train Fraud,Validation Transactions,Validation Fraud,Validation Fraud Rate (%)
0,Fold 1,648337,3649,216113,1300,0.601537
1,Fold 2,864450,4949,216112,1279,0.591823
2,Fold 3,1080562,6228,216113,1278,0.591357


In [13]:
# ==========================================
# STEP 14. 3-Fold 시간순 교차검증
# ==========================================

cv_results = []


for set_name, features in feature_sets.items():

    print("\n" + "=" * 80)
    print(set_name)
    print("=" * 80)

    for fold_name, period in folds.items():

        fold_train, fold_val = get_fold_data(
            df,
            period
        )

        X_train = fold_train[
            features
        ]

        y_train = fold_train[
            "is_fraud"
        ]

        X_val = fold_val[
            features
        ]

        y_val = fold_val[
            "is_fraud"
        ]

        model = make_logistic_pipeline(
            features=features,
            C=1.0,
            penalty="l2",
            solver="lbfgs"
        )

        model.fit(
            X_train,
            y_train
        )

        val_prob = model.predict_proba(
            X_val
        )[:, 1]

        metrics = evaluate_binary_model(
            y_true=y_val,
            prob=val_prob,
            threshold=THRESHOLD
        )

        cv_results.append({

            "Feature Set":
                set_name,

            "Fold":
                fold_name,

            "Threshold":
                THRESHOLD,

            **metrics
        })

        print(
            f"{fold_name} | "
            f"PR-AUC={metrics['PR-AUC']:.6f} | "
            f"Precision={metrics['Precision']:.6f} | "
            f"Recall={metrics['Recall']:.6f} | "
            f"F1={metrics['F1']:.6f}"
        )


Set 1


Fold 1 | PR-AUC=0.440815 | Precision=0.305849 | Recall=0.756154 | F1=0.435534


Fold 2 | PR-AUC=0.432581 | Precision=0.328236 | Recall=0.755278 | F1=0.457603


Fold 3 | PR-AUC=0.447034 | Precision=0.342715 | Recall=0.752739 | F1=0.470991

Set 2


Fold 1 | PR-AUC=0.441505 | Precision=0.307932 | Recall=0.761538 | F1=0.438538


...

Fold 2 | PR-AUC=0.407099 | Precision=0.292819 | Recall=0.736513 | F1=0.419039


Fold 3 | PR-AUC=0.420305 | Precision=0.305008 | Recall=0.738654 | F1=0.431740

Set 5


Fold 1 | PR-AUC=0.518454 | Precision=0.392495 | Recall=0.804615 | F1=0.527617


Fold 2 | PR-AUC=0.511578 | Precision=0.392012 | Recall=0.790461 | F1=0.524106


Fold 3 | PR-AUC=0.523507 | Precision=0.409146 | Recall=0.798122 | F1=0.540971


In [14]:
cv_results_df = pd.DataFrame(
    cv_results
)

display(
    cv_results_df
)

,Feature Set,Fold,Threshold,PR-AUC,ROC-AUC,Accuracy,Precision,Recall,F1,FP,FN,TP,TN
0,Set 1,Fold 1,0.900000,0.440815,0.966848,0.988210,0.305849,0.756154,0.435534,2231,317,983,212582
1,Set 1,Fold 2,0.900000,0.432581,0.970244,0.989404,0.328236,0.755278,0.457603,1977,313,966,212856
2,Set 1,Fold 3,0.900000,0.447034,0.969758,0.990001,0.342715,0.752739,0.470991,1845,316,962,212990
3,Set 2,Fold 1,0.900000,0.441505,0.967468,0.988270,0.307932,0.761538,0.438538,2225,310,990,212588
4,Set 2,Fold 2,0.900000,0.435267,0.970928,0.989496,0.331520,0.762314,0.462085,1966,304,975,212867
5,Set 2,Fold 3,0.900000,0.448995,0.970406,0.990126,0.347252,0.761346,0.476961,1829,305,973,213006
6,Set 3,Fold 1,0.900000,0.440607,0.967034,0.988349,0.308851,0.756923,0.438698,2202,316,984,212611
7,Set 3,Fold 2,0.900000,0.431973,0.970181,0.989385,0.327908,0.756059,0.457427,1982,312,967,212851
8,Set 3,Fold 3,0.900000,0.446290,0.969639,0.990047,0.343828,0.751956,0.471888,1834,317,961,213001
9,Set 4,Fold 1,0.900000,0.416235,0.964717,0.986229,0.266834,0.737692,0.391908,2635,341,959,212178


In [15]:
# ==========================================
# STEP 15. 3-Fold 평균 / 표준편차
# ==========================================

cv_summary_df = (
    cv_results_df
    .groupby(
        "Feature Set"
    )
    .agg(

        Mean_PR_AUC=(
            "PR-AUC",
            "mean"
        ),

        Std_PR_AUC=(
            "PR-AUC",
            "std"
        ),

        Mean_ROC_AUC=(
            "ROC-AUC",
            "mean"
        ),

        Mean_Precision=(
            "Precision",
            "mean"
        ),

        Mean_Recall=(
            "Recall",
            "mean"
        ),

        Mean_F1=(
            "F1",
            "mean"
        ),

        Total_FP=(
            "FP",
            "sum"
        ),

        Total_FN=(
            "FN",
            "sum"
        ),

        Total_TP=(
            "TP",
            "sum"
        ),

        Total_TN=(
            "TN",
            "sum"
        )
    )
    .reset_index()
)


# ==========================================
# 순위 기준
# 1. Mean PR-AUC 높은 순
# 2. Std PR-AUC 낮은 순
# 3. Mean F1 높은 순
# ==========================================

cv_summary_df = (
    cv_summary_df
    .sort_values(
        by=[
            "Mean_PR_AUC",
            "Std_PR_AUC",
            "Mean_F1"
        ],
        ascending=[
            False,
            True,
            False
        ]
    )
    .reset_index(drop=True)
)


cv_summary_df.insert(
    0,
    "Rank",
    range(
        1,
        len(cv_summary_df) + 1
    )
)


display(
    cv_summary_df[
        [
            "Rank",
            "Feature Set",
            "Mean_PR_AUC",
            "Std_PR_AUC",
            "Mean_Precision",
            "Mean_Recall",
            "Mean_F1",
            "Total_FP",
            "Total_FN"
        ]
    ]
)

,Rank,Feature Set,Mean_PR_AUC,Std_PR_AUC,Mean_Precision,Mean_Recall,Mean_F1,Total_FP,Total_FN
0,1,Set 5,0.517847,0.005988,0.397884,0.797733,0.530898,4660,780
1,2,Set 2,0.441922,0.006873,0.328901,0.761733,0.459195,6020,919
2,3,Set 1,0.440143,0.007250,0.325600,0.754723,0.454709,6053,946
3,4,Set 3,0.439623,0.007209,0.326862,0.754980,0.456004,6018,945
4,5,Set 4,0.414547,0.006763,0.288220,0.737620,0.414229,7061,1012


In [16]:
# ==========================================
# STEP 16-1. 최적 Feature Set 선택
# ==========================================

best_set_name = (
    cv_summary_df
    .iloc[0][
        "Feature Set"
    ]
)

best_features = (
    feature_sets[
        best_set_name
    ]
)

print(
    "최적 Feature Set:",
    best_set_name
)

print(
    "변수 개수:",
    len(best_features)
)

print(
    "변수:",
    best_features
)

최적 Feature Set: Set 5
변수 개수: 19
변수: ['category', 'amt', 'is_online', 'recent_24h_high_amt_count', 'category_recent_fraud_rate', 'speed_2', 'customer_mean_amt', 'customer_std_amt', 'amt_ratio_to_mean', 'amt_zscore_card', 'customer_transaction_count', 'trans_hour', 'age', 'rolling_sum_amt_1h', 'prior_normal_median_amt', 'amt_to_prior_median_ratio', 'risk_time_22_04', 'interact_repeat_category', 'has_prior_normal_transaction']


In [19]:
# ==========================================
# STEP 16-2. Fold 3 Train으로 최종 계수 확인
# ==========================================

fold3_train, fold3_val = get_fold_data(
    df,
    folds["Fold 3"]
)

X_train_best = fold3_train[
    best_features
]

y_train_best = fold3_train[
    "is_fraud"
]


best_model = make_logistic_pipeline(
    features=best_features,
    C=1.0,
    penalty="l2",
    solver="lbfgs"
)


best_model.fit(
    X_train_best,
    y_train_best
);

In [20]:
# ==========================================
# STEP 16-3. 전처리 후 변수명
# ==========================================

feature_names = (
    best_model
    .named_steps[
        "preprocessor"
    ]
    .get_feature_names_out()
)


coefficients = (
    best_model
    .named_steps[
        "model"
    ]
    .coef_[0]
)

In [21]:
coef_df = pd.DataFrame({

    "Feature":
        feature_names,

    "Coefficient":
        coefficients
})


# Odds Ratio
coef_df[
    "Odds Ratio"
] = np.exp(
    coef_df[
        "Coefficient"
    ]
)


# 계수 절댓값
coef_df[
    "Abs Coefficient"
] = (
    coef_df[
        "Coefficient"
    ]
    .abs()
)


# 영향력이 큰 순서
coef_df = (
    coef_df
    .sort_values(
        "Abs Coefficient",
        ascending=False
    )
    .reset_index(drop=True)
)


display(
    coef_df.head(30)
)

,Feature,Coefficient,Odds Ratio,Abs Coefficient
0,cat__category_gas_transport,3.094408,22.074168,3.094408
1,cat__category_grocery_pos,2.202565,9.048191,2.202565
2,cat__category_grocery_net,2.199575,9.021179,2.199575
3,cat__category_shopping_net,-1.936715,0.144177,1.936715
4,cat__category_misc_pos,1.677043,5.349715,1.677043
5,cat__category_personal_care,1.526811,4.603475,1.526811
6,cat__category_kids_pets,1.255815,3.510698,1.255815
7,cat__category_shopping_pos,-1.153584,0.315504,1.153584
8,num__risk_time_22_04,1.040620,2.830971,1.040620
9,cat__category_food_dining,1.022407,2.779879,1.022407


In [22]:
print(
    "===== Fraud 방향 TOP 15 ====="
)

fraud_direction = (
    coef_df[
        coef_df[
            "Coefficient"
        ] > 0
    ]
    .sort_values(
        "Coefficient",
        ascending=False
    )
    .head(15)
)

display(
    fraud_direction
)

===== Fraud 방향 TOP 15 =====


,Feature,Coefficient,Odds Ratio,Abs Coefficient
0,cat__category_gas_transport,3.094408,22.074168,3.094408
1,cat__category_grocery_pos,2.202565,9.048191,2.202565
2,cat__category_grocery_net,2.199575,9.021179,2.199575
4,cat__category_misc_pos,1.677043,5.349715,1.677043
5,cat__category_personal_care,1.526811,4.603475,1.526811
6,cat__category_kids_pets,1.255815,3.510698,1.255815
8,num__risk_time_22_04,1.040620,2.830971,1.040620
9,cat__category_food_dining,1.022407,2.779879,1.022407
10,cat__category_travel,0.987490,2.684489,0.987490
11,cat__category_health_fitness,0.965851,2.627023,0.965851


In [23]:
print(
    "===== Normal 방향 TOP 15 ====="
)

normal_direction = (
    coef_df[
        coef_df[
            "Coefficient"
        ] < 0
    ]
    .sort_values(
        "Coefficient",
        ascending=True
    )
    .head(15)
)

display(
    normal_direction
)

===== Normal 방향 TOP 15 =====


,Feature,Coefficient,Odds Ratio,Abs Coefficient
3,cat__category_shopping_net,-1.936715,0.144177,1.936715
7,cat__category_shopping_pos,-1.153584,0.315504,1.153584
13,cat__category_misc_net,-0.872954,0.417716,0.872954
16,num__customer_transaction_count,-0.396524,0.672654,0.396524
20,num__interact_repeat_category,-0.176592,0.838122,0.176592
24,num__has_prior_normal_transaction,-0.132581,0.875832,0.132581
26,num__customer_std_amt,-0.074698,0.928023,0.074698
29,num__prior_normal_median_amt,-0.008786,0.991252,0.008786
30,num__amt_zscore_card,-0.000301,0.999699,0.000301
